# divide and conquer
1. 문제를 더 작은 문제로 나누어서 해결하는 접근법
2. 문제를 나눈 후에 여러개의 cpu에서 동시에 처리되도록 함으로써 계산 시간을 단축시킬 수 있음

In [40]:
import pandas as pd

In [2]:
#대량의 데이터를 읽어올때는... usecols를 이용해서 필요한 컬럼만 읽어옵니다.
df = pd.read_csv('zinc_db/AAAA.txt', sep='\t', usecols=['smiles','logp'])

In [3]:
df

,smiles,logp
0,CO[C@H]1OC[C@@H](O)[C@H](O)[C@H]1O,-1.928
1,C[S@@](=O)CC(N)=O,-1.150
2,NC(=O)N[C@@H]1NC(=O)NC1=O,-2.180
3,O=c1[nH]c(O)c(O)c(=O)[nH]1,-1.526
4,NC(=O)CN1CCC(N)CC1,-1.105
...,...,...
3725,N=C(N)NCCC[C@H](N)C(N)=O,-1.938
3726,CNC(=O)COCCN,-1.292
3727,Nc1n[nH]c(N)n1,-1.031
3728,NCC(N)(CO)CO,-2.373


In [52]:
import glob

# 패턴에 매칭되는 파일 경로를 리스트로 가져옴. (저는 샘플로 100개만 다운 받아봤어요)
# 별표(*)는 모든 경우르 의미합니다. *.txt면 모든 .txt 파일을 선택합니다.
txt_files = glob.glob('./zinc_db/*.txt')

print(f"매칭된 파일 개수: {len(txt_files)}")
# 상위 5개 파일명만 출력해서 확인
print("파일 샘플:", txt_files[-5:])

매칭된 파일 개수: 100
파일 샘플: ['./zinc_db\\FAED.txt', './zinc_db\\GAAA.txt', './zinc_db\\GAAB.txt', './zinc_db\\GAAC.txt', './zinc_db\\GAAD.txt']


In [56]:
# 패턴에 매칭되는 파일 경로를 리스트로 가져오는데,
# 파일 이름이 A로 시작하는 파일만 선택합니다. (A*.txt 패턴을 보이는 파일만 선택)
# 현재 각 학생별로 다운 받은 파일 수가 다르기 때문에.. 일단 A로 시작하는 파일만 선택해서 실험해보겠습니다.

txt_files = glob.glob('./zinc_db/A*.txt')

print(f"매칭된 파일 개수: {len(txt_files)}")
# 상위 5개 파일명만 출력해서 확인
print("파일 샘플:", txt_files[-5:])

매칭된 파일 개수: 16
파일 샘플: ['./zinc_db\\AACD.txt', './zinc_db\\AAEA.txt', './zinc_db\\AAEB.txt', './zinc_db\\AAEC.txt', './zinc_db\\AAED.txt']


In [51]:
# 각 txt 파일들에 들어있는 물질 개수를 확인해봤습니다. 파일마다 제각각입니다.
for f in txt_files:
    df = pd.read_csv(f, sep='\t', usecols=['smiles','logp'])
    print (df.shape)

(3730, 2)
(4107, 2)
(1012, 2)
(16545, 2)
(21, 2)
(65, 2)
(12, 2)
(74, 2)
(13, 2)
(41, 2)
(15, 2)
(9, 2)
(270, 2)
(590, 2)
(133, 2)
(2170, 2)


In [10]:
df

,smiles,logp
0,C[C@H](C(=O)c1c(N)n(C)c(=O)n(C)c1=O)N1CCCN(S(C...,-1.795
1,CCS(=O)(=O)NCCCNC(=O)[C@H]1CC(=O)N([C@@H]2CCS(...,-1.532
2,CCN1CCN(CC(=O)NCC(=O)N(C)CCc2ccccn2)C(=O)C1=O,-1.111
3,C[C@H]1CN(C(=O)NCCNS(C)(=O)=O)CCN1c1nccn2cnnc12,-1.106
4,COc1ccc(NC(=O)C(=O)N2CCN3C(=O)C(=O)NC[C@@H]3C2...,-1.189
...,...,...
263657,O=C1CC[C@H](C(=O)N2CC3(CCCN3C(=O)c3cc(=O)n4[nH...,-1.239
263658,Cc1ccc(=O)n(C(C)(C)C(=O)N2CCCC23CN(C(=O)[C@@H]...,-1.224
263659,CC[C@@H](NC(N)=O)C(=O)N1CC2(CCCN2C(=O)Cn2ccncc...,-1.106
263660,O=C(CN(CCO)CCO)N1CC2(CCCN2C(=O)CCc2cnccn2)C1,-1.101


In [11]:
#df의 logp컬럼에서 최소값을 확인하는 방법
logp_min = df['logp'].min()
logp_min

np.float64(-6.881)

In [12]:
#logp 값이 가장 작은 smiles 코드 찾는 방법
df_low = df.loc[df['logp']==logp_min,'smiles']
df_low

148029    NCCN(C[C@H](O)[C@@H](O)[C@H](O)[C@H](O)CO)C[C@...
Name: smiles, dtype: str

In [ ]:
type(df_low)

In [ ]:
# pandas Series에서 필요한 smiles 코드 정보만 추출하는 방법
df.loc[df['logp']==logp_min,'smiles'].item()

In [13]:
# pandas Series에서 필요한 smiles 코드 정보만 리스트로 추출하는 방법
df.loc[df['logp']==logp_min,'smiles'].to_list()

['NCCN(C[C@H](O)[C@@H](O)[C@H](O)[C@H](O)CO)C[C@H](O)[C@@H](O)[C@H](O)[C@H](O)CO']

In [14]:
#리스트에 있는 문자를 하나의 str으로 묶는 방법
', '.join(['A','B','C'])

'A, B, C'

In [15]:
#만약에 최소값이 smiles 코드가 여러개라면? list로 추출한 후에 join함수를 이용해서 하나의 str으로 묶을 수 있음.
', '.join(df.loc[df['logp']==logp_min,'smiles'].to_list())

'NCCN(C[C@H](O)[C@@H](O)[C@H](O)[C@H](O)CO)C[C@H](O)[C@@H](O)[C@H](O)[C@H](O)CO'

# Quiz1
1. 가장 단순한 검색 방법
2. 각 파일에서 가장 작은 logp 값을 찾는다.
3. 그 다음 파일에서 더 작은 logp 값을 찾는다면, 더 작은 logp 값으로 새로 저장한다.
4. 모든 파일을 확인하면서 더 작은 logp 값이 있는지 확인해본다.

In [57]:
import time

start_time = time.time()

lowest_logp = float('inf') #무한대를 의미. 초기 값을 가장 큰 값으로 선택.
lowest_smiles = '' #smiles코드는 그냥 빈 str
for f in txt_files:
    df = pd.read_csv(f, sep='\t', usecols=['smiles','logp'])
    logp_min = df['logp'].min()
    smi_min = ', '.join(df.loc[df['logp']==logp_min,'smiles'].to_list())
    if logp_min < lowest_logp:
        lowest_logp = logp_min
        lowest_smi = smi_min

end_time = time.time()
print ('smiles:', lowest_smi)
print ('logp:', lowest_logp)
print ('total computation time:', end_time - start_time)

smiles: [O-][I+3]([O-])([O-])O
logp: -7.12
total computation time: 0.15156102180480957


In [22]:
#https://www.dask.org/

import dask.dataframe as dd
from dask.distributed import Client

# 1. 로컬 클러스터 시작 (주피터에서 대시보드로 현황 모니터링 가능)
client = Client(n_workers=3, threads_per_worker=6) 

client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 3
Total threads: 18,Total memory: 15.48 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:62842,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:62855,Total threads: 6
Dashboard: http://127.0.0.1:62860/status,Memory: 5.16 GiB
Nanny: tcp://127.0.0.1:62845,


In [32]:
# 2. 2,000개 파일을 하나의 가상 데이터프레임으로 연결 (Lazy Loading)
# 파일명이 'zinc_*.csv' 형태라면 와일드카드로 한 번에 로드

start_time = time.time()
df = dd.read_csv('./zinc_db/A*.txt', sep='\t', usecols=['smiles', 'logp'])
end_time = time.time()

# 전체 행 개수 확인 (실제 데이터를 스캔하므로 시간이 걸릴 수 있음)
row_count = len(df) 
print(f"Total rows: {row_count:,}")
print ("Time spent:", end_time - start_time)

Total rows: 5,277,078
Time spent: 0.16502714157104492


In [33]:
# 파티션 개수 확인
print(f"Number of partitions: {df.npartitions}")

# 데이터의 스키마(컬럼명, 데이터 타입) 및 지연 상태 확인
print(df)

Number of partitions: 102
Dask DataFrame Structure:
                 smiles     logp
npartitions=102                 
                 string  float64
                    ...      ...
...                 ...      ...
                    ...      ...
                    ...      ...
Dask Name: to_string_dtype, 2 expressions
Expr=ArrowStringConversion(frame=FromMapProjectable(ad3a3f8))


In [34]:
# 데이터의 구조와 타입 확인
df.info()

<class 'dask.dataframe.dask_expr.DataFrame'>
Columns: 2 entries, smiles to logp
dtypes: string(1), float64(1)

In [35]:
# 3. 최솟값 계산 (분할 정복 알고리즘이 내부적으로 실행됨)
# 실제 연산은 .compute()를 호출할 때까지 시작되지 않음

start_time = time.time()
min_logp = df['logp'].min().compute()
result = df[df['logp']==min_logp].compute()
end_time = time.time()

print(f"최저 logP 물질: {result}")
print ('time for computation:',end_time-start_time)

최저 logP 물질:                                        smiles   logp
436  OC1(O)C(O)(O)C(O)(O)C(O)(O)C(O)(O)C1(O)O -7.915
time for computation: 16.68193483352661


In [36]:
result['smiles'].item()

'OC1(O)C(O)(O)C(O)(O)C(O)(O)C(O)(O)C1(O)O'

In [37]:
# 각 파티션별 메모리 사용량 (바이트 단위)
memory_usage = df.memory_usage(deep=True).compute()
print(f"Total Memory: {memory_usage.sum() / 1e9:.2f} GB")

Total Memory: 0.34 GB


In [38]:
# 여러 작업을 정의만 해두고
import dask
start_time = time.time()

min_val = df['logp'].min()
min_rows = df[df['logp'] == min_val]

end_time = time.time()

# 한 번의 compute로 모든 결과를 가져옵니다
final_min, final_rows = dask.compute(min_val, min_rows)
print(f"최저 logP 물질: {final_rows}")
print ('time for computation:',end_time-start_time)

최저 logP 물질:                                        smiles   logp
436  OC1(O)C(O)(O)C(O)(O)C(O)(O)C(O)(O)C1(O)O -7.915
time for computation: 0.009720087051391602


In [39]:
client.close()